<a href="https://colab.research.google.com/github/Andysimps0n/Deep-Learning/blob/main/ResNet_scaling_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip

import os

# Move validation images to class-specific subfolders
val_dir = 'tiny-imagenet-200/val'
with open(os.path.join(val_dir, 'val_annotations.txt'), 'r') as f:
    for line in f.readlines():
        split_line = line.split('\t')
        img_name = split_line[0]
        class_name = split_line[1]

        class_dir = os.path.join(val_dir, class_name)
        if not os.path.exists(class_dir):
            os.mkdir(class_dir)

        os.rename(os.path.join(val_dir, 'images', img_name),
                  os.path.join(class_dir, img_name))

# Clean up empty images folder
os.rmdir(os.path.join(val_dir, 'images'))

URL transformed to HTTPS due to an HSTS policy
--2026-05-17 07:19:44--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘tiny-imagenet-200.zip.1’

tiny-imagenet-200.z 100%[===================>] 236.61M  88.5MB/s    in 2.7s    

2026-05-17 07:19:47 (88.5 MB/s) - ‘tiny-imagenet-200.zip.1’ saved [248100043/248100043]

replace tiny-imagenet-200/words.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [24]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

train_transforms = transforms.Compose([
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
      brightness=0.2,
      contrast=0.2,
      saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4802, 0.4481, 0.3975],
        std=[0.2764, 0.2692, 0.2821]
    ),

])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4802, 0.4481, 0.3975],
        std=[0.2764, 0.2692, 0.2821]
    )
])

train_test_set = datasets.ImageFolder('tiny-imagenet-200/train', transform=train_transforms)
start = int(len(train_test_set) * 0.9)
finish = int(len(train_test_set) * 0.1)

val_set = datasets.ImageFolder('tiny-imagenet-200/val', transform=test_transforms)
train_set, test_set = random_split(train_test_set, [start, finish])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)



In [28]:
small_train_size = 30000

small_train_set, _ = random_split(
    train_set,
    [small_train_size, len(train_set) - small_train_size]
)

train_loader = DataLoader(
    small_train_set,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [16]:
from torch.nn.modules import Linear
import torch.nn as nn


class BasicBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super(BasicBlock, self).__init__()
    self.stride = stride

    # convolution to halve the identity
    self.conv_halve = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
    self.bn_halve = nn.BatchNorm2d(out_channels)

    # Layer 1
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)

    # Layer 2
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(out_channels)

  def forward(self, x):
    identity = x

    # Layer 1
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)

    # Layer 2
    x = self.conv2(x)
    x = self.bn2(x)

    # skip connection
    if(self.stride==2):
      identity = self.conv_halve(identity)
      identity = self.bn_halve(identity)
    x += identity
    x = self.relu(x)

    return x


class ResNet(nn.Module):
  def __init__(self):
    super(ResNet, self).__init__()

    self.stem = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            )

    self.features = nn.Sequential(
      BasicBlock(64, 64),
      BasicBlock(64, 64),
      BasicBlock(64, 64),

      BasicBlock(64, 128, 2),
      BasicBlock(128, 128),
      BasicBlock(128, 128),
      BasicBlock(128, 128),

      BasicBlock(128, 256, 2),
      BasicBlock(256, 256),
      BasicBlock(256, 256),
      BasicBlock(256, 256),
      BasicBlock(256, 256),
      BasicBlock(256, 256),

      BasicBlock(256, 512, 2),
      BasicBlock(512, 512),
      BasicBlock(512, 512),

      nn.AdaptiveAvgPool2d((1,1)),
      nn.Flatten(),

      nn.Dropout(0.3),
      nn.Linear(in_features=512, out_features=200),
    )

  def forward(self, x):
    x = self.stem(x)
    x = self.features(x)
    return x

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 3

In [18]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [29]:
import torch

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.1
)

for epoch in range(epochs):


    # Train
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total


    # Validation
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = outputs.max(1)

            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total


    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )

    scheduler.step()



Epoch [1/3] Loss: 1609.9825 Train Acc: 21.87% Val Acc: 23.97%
Epoch [2/3] Loss: 1539.7560 Train Acc: 24.19% Val Acc: 24.80%
Epoch [3/3] Loss: 1485.9968 Train Acc: 26.41% Val Acc: 25.76%


In [37]:
alpha_list = {1.0, 1.1, 1.2, 1.3}
beta_list = {1.0, 1.05, 1.1, 1.15}
gamma_list = {1.0, 1.1, 1.15, 1.2}


combinations = []

for alpha in alpha_list:
  for beta in beta_list:
    for gamma in gamma_list:
      if 2 - (alpha * (beta**2) * (gamma ** 2)) < 0.1: # error rage = 0.12
        combination = (alpha, beta, gamma)
        combinations.append(combination)

print(combinations)
print(len(combinations))

[(1.2, 1.1, 1.2), (1.2, 1.1, 1.15), (1.2, 1.05, 1.2), (1.2, 1.15, 1.2), (1.2, 1.15, 1.15), (1.2, 1.15, 1.1), (1.1, 1.1, 1.2), (1.1, 1.15, 1.2), (1.1, 1.15, 1.15), (1.3, 1.1, 1.2), (1.3, 1.1, 1.15), (1.3, 1.1, 1.1), (1.3, 1.05, 1.2), (1.3, 1.15, 1.2), (1.3, 1.15, 1.15), (1.3, 1.15, 1.1), (1.0, 1.15, 1.2)]
17


In [ ]:
def model_scaled(ratio_combination):


  widths = [64, 128, 256, 512]
  depths = [3, 4, 6, 3]

  # Resolution is fixed
  # resolution = 64


  scaled_widths = widths * ratio_combination[1]
  scaled_depths = depths * ratio_combination[0]

  class ResNet(nn.Module):
    def __init__(self):
      super(ResNet, self).__init__()

      self.stem = nn.Sequential(
          nn.Conv2d(3, widths[0], kernel_size=7, stride=2, padding=3, bias=False),
          nn.BatchNorm2d(widths[0]),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
      )

      self.features = nn.Sequential(
          BasicBlock(widths[0], widths[0]),
          BasicBlock(widths[0], widths[0]),
          BasicBlock(widths[0], widths[0]),

          BasicBlock(widths[0], widths[1], 2),
          BasicBlock(widths[1], widths[1]),
          BasicBlock(widths[1], widths[1]),
          BasicBlock(widths[1], widths[1]),

          BasicBlock(widths[1], widths[2], 2),
          BasicBlock(widths[2], widths[2]),
          BasicBlock(widths[2], widths[2]),
          BasicBlock(widths[2], widths[2]),
          BasicBlock(widths[2], widths[2]),
          BasicBlock(widths[2], widths[2]),

          BasicBlock(widths[2], widths[3], 2),
          BasicBlock(widths[3], widths[3]),
          BasicBlock(widths[3], widths[3]),

          nn.AdaptiveAvgPool2d((1,1)),
          nn.Flatten(),

          nn.Dropout(0.3),
          nn.Linear(in_features=widths[3], out_features=200),
      )

      def forward(self, x):
          x = self.stem(x)
          x = self.features(x)
          return x


  epochs = 5

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = ResNet().to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
  scheduler = torch.optim.lr_scheduler.StepLR(
      optimizer,
      step_size=10,
      gamma=0.1
  )

  for epoch in range(epochs):

    # Train
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_acc = 100. * correct / total


    # Validate
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    val_acc = 100. * val_correct / val_total



    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )
    scheduler.step()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    print(f"Test Accuracy: {100. * correct / total:.2f}%")




In [20]:
print(train_test_set.class_to_idx)
print(val_set.class_to_idx)

{'n01443537': 0, 'n01629819': 1, 'n01641577': 2, 'n01644900': 3, 'n01698640': 4, 'n01742172': 5, 'n01768244': 6, 'n01770393': 7, 'n01774384': 8, 'n01774750': 9, 'n01784675': 10, 'n01855672': 11, 'n01882714': 12, 'n01910747': 13, 'n01917289': 14, 'n01944390': 15, 'n01945685': 16, 'n01950731': 17, 'n01983481': 18, 'n01984695': 19, 'n02002724': 20, 'n02056570': 21, 'n02058221': 22, 'n02074367': 23, 'n02085620': 24, 'n02094433': 25, 'n02099601': 26, 'n02099712': 27, 'n02106662': 28, 'n02113799': 29, 'n02123045': 30, 'n02123394': 31, 'n02124075': 32, 'n02125311': 33, 'n02129165': 34, 'n02132136': 35, 'n02165456': 36, 'n02190166': 37, 'n02206856': 38, 'n02226429': 39, 'n02231487': 40, 'n02233338': 41, 'n02236044': 42, 'n02268443': 43, 'n02279972': 44, 'n02281406': 45, 'n02321529': 46, 'n02364673': 47, 'n02395406': 48, 'n02403003': 49, 'n02410509': 50, 'n02415577': 51, 'n02423022': 52, 'n02437312': 53, 'n02480495': 54, 'n02481823': 55, 'n02486410': 56, 'n02504458': 57, 'n02509815': 58, 'n0266

In [26]:
torch.save(model.state_dict(), "1_default_model.pth")

In [35]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"Test Accuracy: {100. * correct / total:.2f}%")

Test Accuracy: 24.50%


# model 1

default ResNet.
Test accuracy, train accuracy = 20%


#model 2
default Res Net with 30k img ==> 24%

In [34]:
alpha_list = {1.0, 1.1, 1.2, 1.3}
beta_list = {1.0, 1.05, 1.1, 1.15}
gamma_list = {1.0, 1.1, 1.15, 1.2}


combinations = []

for alpha in alpha_list:
  for beta in beta_list:
    for gamma in gamma_list:
      if 2 - (alpha * (beta**2) * (gamma ** 2)) < 0.1: # error rage = 0.12
        combination = {
            "alpha" : alpha,
            "beta" : beta,
            "gamma" : gamma
        }
        combinations.append(combination)

print(combinations)
print(len(combinations))

[{'alpha': 1.2, 'beta': 1.1, 'gamma': 1.2}, {'alpha': 1.2, 'beta': 1.1, 'gamma': 1.15}, {'alpha': 1.2, 'beta': 1.05, 'gamma': 1.2}, {'alpha': 1.2, 'beta': 1.15, 'gamma': 1.2}, {'alpha': 1.2, 'beta': 1.15, 'gamma': 1.15}, {'alpha': 1.2, 'beta': 1.15, 'gamma': 1.1}, {'alpha': 1.1, 'beta': 1.1, 'gamma': 1.2}, {'alpha': 1.1, 'beta': 1.15, 'gamma': 1.2}, {'alpha': 1.1, 'beta': 1.15, 'gamma': 1.15}, {'alpha': 1.3, 'beta': 1.1, 'gamma': 1.2}, {'alpha': 1.3, 'beta': 1.1, 'gamma': 1.15}, {'alpha': 1.3, 'beta': 1.1, 'gamma': 1.1}, {'alpha': 1.3, 'beta': 1.05, 'gamma': 1.2}, {'alpha': 1.3, 'beta': 1.15, 'gamma': 1.2}, {'alpha': 1.3, 'beta': 1.15, 'gamma': 1.15}, {'alpha': 1.3, 'beta': 1.15, 'gamma': 1.1}, {'alpha': 1.0, 'beta': 1.15, 'gamma': 1.2}]
17
